# Remesher Demo 03 — Bring Your Own Image + Mixamo FBX

Run `00_comfy3d_setup_models.ipynb` first to configure Comfy3D, the shared model cache, and `config.json`.

Use Jupyter browse/upload widgets to provide your own character image and optional Mixamo FBX animation. The notebook saves uploads into `/workspace/input/uploads`, then runs image-to-GLB, rigging, head cleanup, and retargeting.

In [ ]:
import json, os, pathlib, subprocess
WORKSPACE = pathlib.Path('/workspace')
ROOT = pathlib.Path('/workspace/remesher')
INPUT = pathlib.Path('/workspace/input')
OUTPUT = pathlib.Path('/workspace/output')
MODELS = pathlib.Path('/workspace/models')
for p in [INPUT, OUTPUT, MODELS]:
    p.mkdir(parents=True, exist_ok=True)
COMFY_CONTAINER = os.environ.get('COMFY_CONTAINER', 'remesher-comfy3d')
CONFIG_PATH = ROOT / 'config.json'
if not CONFIG_PATH.exists():
    raise FileNotFoundError('Missing /workspace/remesher/config.json. Run 00_comfy3d_setup_models.ipynb first.')
config = json.loads(CONFIG_PATH.read_text())
print('ROOT', ROOT); print('INPUT', INPUT); print('OUTPUT', OUTPUT); print('MODELS', MODELS)
print('ComfyUI server_url:', config.get('server_url'))
print('If this fails later because Comfy3D or models are not ready, run notebook 00 first.')


In [ ]:
subprocess.run(['comfy-prompt-cli', '--help'], cwd=ROOT, check=True)
subprocess.run(['comfy-prompt-cli', 'health', '--config', 'config.json'], cwd=ROOT, check=True)


In [ ]:
from pathlib import Path
import time
from urllib.parse import quote
import uuid
import html
from IPython.display import Image as IPImage, display, HTML


def display_glb_model_viewer(glb_path, notebook_root="/workspace", height=560):
    """Display a local GLB using <model-viewer> in an iframe.

    JupyterLab may not execute <script> tags inserted directly by HTML(...),
    which leaves a direct notebook output blank. An iframe srcdoc gives the
    viewer its own page where module scripts execute normally. The GLB is served
    by Jupyter's /files/ endpoint instead of being base64-inlined.
    """
    glb_path = Path(glb_path).resolve()
    notebook_root = Path(notebook_root).resolve()

    if not glb_path.exists():
        raise FileNotFoundError(glb_path)

    size_mb = glb_path.stat().st_size / 1024 / 1024
    print(f"GLB: {glb_path}")
    print(f"Size: {size_mb:.1f} MB")

    try:
        rel = glb_path.relative_to(notebook_root).as_posix()
    except ValueError:
        display(HTML(f"""
<div style="padding:0.75rem; border:1px solid #d99; background:#fff6f6; border-radius:8px;">
  <b>Preview path is outside notebook_root.</b><br>
  GLB exists, but Jupyter may not be able to serve it through <code>/files/</code>.<br>
  Path: <code>{glb_path}</code><br>
  notebook_root: <code>{notebook_root}</code>
</div>
"""))
        return

    files_url = "/files/" + quote(rel)
    # model-viewer must be served from Jupyter root (/workspace). The repo copy
    # lives under /workspace/remesher/workspace/vendor; copy it to /workspace/vendor
    # if needed so /files/vendor/model-viewer.min.js does not 404.
    vendor_src = Path('/workspace/remesher/workspace/vendor/model-viewer.min.js')
    vendor_dst = Path('/workspace/vendor/model-viewer.min.js')
    try:
        if vendor_src.exists() and not vendor_dst.exists():
            vendor_dst.parent.mkdir(parents=True, exist_ok=True)
            vendor_dst.write_bytes(vendor_src.read_bytes())
    except Exception as exc:
        print(f'Warning: could not stage model-viewer asset: {exc}')
    model_viewer_js_url = '/files/vendor/model-viewer.min.js' if vendor_dst.exists() else 'https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js'
    viewer_id = f"mv-{uuid.uuid4().hex}"
    title = glb_path.name

    display(HTML(f"""
<div style="font-family:system-ui,-apple-system,Segoe UI,sans-serif; margin:0.5rem 0;">
  <div><b>GLB file:</b> <a href="{files_url}" target="_blank" rel="noopener">{files_url}</a></div>
  <div><b>Size:</b> {size_mb:.1f} MB</div>
</div>
"""))

    iframe_doc = f"""<!doctype html>
<html>
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <script type=\"module\" src=\"{model_viewer_js_url}\"></script>
  <style>
    html, body {{ margin: 0; width: 100%; height: 100%; background: #111; color: #ddd; font-family: system-ui, -apple-system, Segoe UI, sans-serif; }}
    .bar {{ box-sizing: border-box; padding: 8px 10px; background: #181818; border-bottom: 1px solid #333; font-size: 13px; }}
    model-viewer {{ width: 100%; height: calc(100% - 38px); background: #111; }}
    a {{ color: #8ab4ff; }}
  </style>
</head>
<body>
  <div class=\"bar\">
    <b>3D preview:</b> {html.escape(title)} · <span id=\"status\">loading</span> ·
    <a href=\"{files_url}\" target=\"_blank\" rel=\"noopener\">open GLB</a>
  </div>
  <model-viewer id=\"{viewer_id}\" src=\"{files_url}\" camera-controls auto-rotate autoplay animation-crossfade-duration=\"300\" shadow-intensity=\"1\" exposure=\"1\" ar></model-viewer>
  <script>
    const viewer = document.getElementById({json.dumps(viewer_id)});
    const status = document.getElementById('status');
    viewer.addEventListener('load', () => {{
      const animations = viewer.availableAnimations || [];
      status.textContent = animations.length ? `loaded · animations: ${{animations.join(', ')}} · playing` : 'loaded · no animations found';
      if (animations.length) {{
        viewer.animationName = animations[0];
        const maybePromise = viewer.play && viewer.play();
        if (maybePromise && maybePromise.catch) maybePromise.catch((err) => console.warn('model-viewer autoplay failed', err));
      }}
    }});
    viewer.addEventListener('error', (event) => {{
      status.textContent = 'error loading GLB — open the GLB link or browser console for details';
      console.error('model-viewer failed for {files_url}', event);
    }});
    setTimeout(() => {{
      if (status.textContent === 'loading') status.textContent = 'still loading; large GLBs may take a while';
    }}, 5000);
  </script>
</body>
</html>"""

    display(HTML(f"""
<iframe
  srcdoc="{html.escape(iframe_doc, quote=True)}"
  style="width:100%; height:{height}px; border:1px solid #ddd; border-radius:8px; background:#111;"
  sandbox="allow-scripts allow-same-origin allow-popups allow-downloads">
</iframe>
"""))


## Upload your source image and Mixamo FBX


In [ ]:
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

UPLOAD_DIR = INPUT / 'uploads'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

image_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='Browse image')
fbx_upload = widgets.FileUpload(accept='.fbx', multiple=False, description='Browse FBX')
status = widgets.Output()

def _first_upload_value(upload):
    value = upload.value
    if isinstance(value, dict):
        return next(iter(value.items())) if value else None
    if isinstance(value, (list, tuple)):
        if not value:
            return None
        item = value[0]
        return (item.get('name'), item) if isinstance(item, dict) else None
    return None

def save_uploads(_=None):
    global USER_IMAGE_PATH, USER_FBX_PATH
    with status:
        status.clear_output()
        image_item = _first_upload_value(image_upload)
        if image_item is None:
            print('Choose an image first.')
            return
        image_name, image_info = image_item
        USER_IMAGE_PATH = UPLOAD_DIR / image_name
        USER_IMAGE_PATH.write_bytes(bytes(image_info['content']))
        print('Saved image:', USER_IMAGE_PATH)
        fbx_item = _first_upload_value(fbx_upload)
        if fbx_item is not None:
            fbx_name, fbx_info = fbx_item
            USER_FBX_PATH = UPLOAD_DIR / fbx_name
            USER_FBX_PATH.write_bytes(bytes(fbx_info['content']))
            print('Saved FBX:', USER_FBX_PATH)
        else:
            USER_FBX_PATH = INPUT / 'animation_templates' / 'mixamo' / 'Mma_Kick.fbx'
            print('No FBX uploaded; using default:', USER_FBX_PATH)

save_button = widgets.Button(description='Save uploads', button_style='success')
save_button.on_click(save_uploads)
display(widgets.VBox([
    widgets.HTML('<b>Upload character image</b>'), image_upload,
    widgets.HTML('<b>Optional: upload Mixamo FBX for retargeting</b>'), fbx_upload,
    save_button,
    status,
]))


## Preview uploaded inputs


In [ ]:
from IPython.display import Image as IPImage, display
from pathlib import Path

if 'USER_IMAGE_PATH' not in globals():
    raise NameError('Upload an image and click Save uploads first.')
print('User image:', USER_IMAGE_PATH)
display(IPImage(filename=str(USER_IMAGE_PATH)))
print('Retarget FBX:', USER_FBX_PATH)


## Image to GLB


In [ ]:
import subprocess, time
from pathlib import Path

IMAGE_TO_GLB_TARGET_FACE_NUM = int(globals().get('IMAGE_TO_GLB_TARGET_FACE_NUM_OVERRIDE', 80000))
print('Image-to-GLB target face num:', IMAGE_TO_GLB_TARGET_FACE_NUM)
out_dir = OUTPUT / 'glbs'; out_dir.mkdir(parents=True, exist_ok=True)
run_started = time.time()
output_prefix = f'user_upload_{Path(USER_IMAGE_PATH).stem}_{IMAGE_TO_GLB_TARGET_FACE_NUM}'
subprocess.run([
    'comfy-prompt-cli', 'image-to-glb',
    '--config', 'config.json',
    '--image', str(USER_IMAGE_PATH),
    '--target-face-num', str(IMAGE_TO_GLB_TARGET_FACE_NUM),
    '--filename-prefix', output_prefix,
    '--out-dir', str(out_dir),
    '--timeout', '7200',
    '--verbose',
], cwd=ROOT, check=True)
outputs = sorted([p for p in out_dir.glob(f'{output_prefix}*.glb') if p.stat().st_mtime >= run_started - 2], key=lambda p: p.stat().st_mtime)
if not outputs:
    outputs = sorted(out_dir.glob(f'{output_prefix}*.glb'), key=lambda p: p.stat().st_mtime)
GENERATED_GLB_PATH = outputs[-1]
print('Generated GLB:', GENERATED_GLB_PATH)
display_glb_model_viewer(GENERATED_GLB_PATH, notebook_root=Path('/workspace'))


## Rig GLB and clean head weights


In [ ]:
import subprocess
from pathlib import Path

mesh_path = Path(GENERATED_GLB_PATH)
RIG_TARGET_FACE_COUNT = int(globals().get('RIG_TARGET_FACE_COUNT_OVERRIDE', 80000))
out_dir = OUTPUT / 'rigged'; out_dir.mkdir(parents=True, exist_ok=True)
rig_name = f'{mesh_path.stem}_rigged'
subprocess.run([
    'comfy-prompt-cli', 'rig-glb',
    '--config', 'config.json',
    '--mesh', str(mesh_path),
    '--glb-name', rig_name,
    '--out-dir', str(out_dir),
    '--target-face-count', str(RIG_TARGET_FACE_COUNT),
    '--embed-textures',
    '--timeout', '7200',
    '--verbose',
], cwd=ROOT, check=True)
RAW_RIGGED_GLB_PATH = sorted(out_dir.glob(f'{rig_name}*.glb'), key=lambda p: p.stat().st_mtime)[-1]
CLEANED_RIGGED_GLB_PATH = out_dir / f'{RAW_RIGGED_GLB_PATH.stem}_headfix.glb'
subprocess.run([
    'comfy-prompt-cli', 'skin-cleanup-glb',
    '--input-glb', str(RAW_RIGGED_GLB_PATH),
    '--output-name', CLEANED_RIGGED_GLB_PATH.stem,
    '--out-dir', str(out_dir),
    '--mode', 'conservative',
    '--repair-zones', 'head_top,head_neck',
    '--worker-file', str(ROOT / 'docker' / 'demo-jupyter' / 'scripts' / 'anatomical_cleanup_worker.py'),
], cwd=ROOT, check=True)
GENERATED_RIGGED_GLB_PATH = CLEANED_RIGGED_GLB_PATH
print('Cleaned/head-fixed rigged GLB:', GENERATED_RIGGED_GLB_PATH)
display_glb_model_viewer(GENERATED_RIGGED_GLB_PATH, notebook_root=Path('/workspace'))


## Retarget uploaded/default Mixamo FBX


In [ ]:
import subprocess
from pathlib import Path

source_rigged = Path(GENERATED_RIGGED_GLB_PATH)
animation_fbx = Path(USER_FBX_PATH)
if not animation_fbx.exists():
    raise FileNotFoundError(animation_fbx)
ANIMATED_OUTPUT_NAME = f'{source_rigged.stem}_{animation_fbx.stem}'
subprocess.run([
    'comfy-prompt-cli', 'retarget-glb',
    '--rigged-glb', str(source_rigged),
    '--animation', str(animation_fbx),
    '--glb-name', ANIMATED_OUTPUT_NAME,
    '--out-dir', str(OUTPUT / 'comfyui' / 'animated'),
    '--worker-file', str(ROOT / 'docker' / 'demo-jupyter' / 'scripts' / 'arp_retarget_worker.py'),
], cwd=ROOT, check=True)
ANIMATED_GLB_PATH = OUTPUT / 'comfyui' / 'animated' / f'{ANIMATED_OUTPUT_NAME}.glb'
print('Animated GLB:', ANIMATED_GLB_PATH)
display_glb_model_viewer(ANIMATED_GLB_PATH, notebook_root=Path('/workspace'), height=640)
